# NOTEBOOK : 01_player_preprocessing.ipynb
# ROLE     : Harmonize player keys, standardize positions & team names, and export clean player/team CSVs.
# INPUT    : ../../data/raw/<season>/master_gameweeks.csv, master_player_history.csv, ../artifacts/team_mapping.json
# OUTPUT   : ../../data/processed/{teams.csv, players.csv, player_gw.csv, player_history.csv}

In [1]:
from pathlib import Path
import json
import unicodedata
import pandas as pd

RAW_DATA_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
MAPPING_FILE = Path("../artifacts/team_mapping.json")

SEASONS = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26", "2026-27"]
VALID_POSITIONS = {"GK", "DEF", "MID", "FWD"}

with open(MAPPING_FILE, "r", encoding="utf-8") as f:
    TEAM_MAPPING = json.load(f)


def normalize_player_key(name):
    """Lowercase, strip whitespace, and remove accents for a stable player_key."""
    if pd.isna(name):
        return ""
    text = str(name).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    return " ".join(text.split())


def standardize_position(pos):
    """Map position variants (e.g. GKP -> GK) to standard GK/DEF/MID/FWD."""
    if pd.isna(pos):
        return "MID"
    p = str(pos).strip().upper()
    if p in ("GK", "GKP", "GOALKEEPER"):
        return "GK"
    if p in ("DEF", "DEFENDER"):
        return "DEF"
    if p in ("FWD", "FORWARD", "ST"):
        return "FWD"
    return "MID"


print("Helpers loaded.")

Helpers loaded.


In [2]:
gw_frames = []
KEEP_GW_COLS = [
    "name", "position", "team", "element", "round", "kickoff_time",
    "was_home", "opponent_team", "minutes", "goals_scored", "assists",
    "clean_sheets", "goals_conceded", "bps", "ict_index", "xP",
    "total_points", "value", "yellow_cards", "red_cards",
]

for season in SEASONS:
    gw_path = RAW_DATA_DIR / season / "master_gameweeks.csv"
    if not gw_path.exists():
        print(f"[SKIP] Missing {gw_path}")
        continue

    df_gw = pd.read_csv(gw_path, low_memory=False)
    available_cols = [c for c in KEEP_GW_COLS if c in df_gw.columns]
    df_gw = df_gw[available_cols].copy()
    df_gw["season"] = season

    df_gw["name"] = df_gw["name"].astype(str).str.strip()
    df_gw["player_key"] = df_gw["name"].apply(normalize_player_key)
    df_gw["position"] = df_gw["position"].apply(standardize_position)
    df_gw["team"] = df_gw["team"].astype(str).str.strip().replace(TEAM_MAPPING)

    # Ensure non-negative minutes
    df_gw["minutes"] = pd.to_numeric(df_gw["minutes"], errors="coerce").fillna(0).clip(lower=0).astype(int)
    df_gw["total_points"] = pd.to_numeric(df_gw["total_points"], errors="coerce").fillna(0).astype(int)

    gw_frames.append(df_gw)
    print(f"Loaded {season} gameweeks: {len(df_gw)} rows")

player_gw = pd.concat(gw_frames, ignore_index=True)
player_gw.to_csv(PROCESSED_DIR / "player_gw.csv", index=False)
print(f"\nSaved player_gw.csv: {len(player_gw)} rows")

Loaded 2020-21 gameweeks: 24365 rows
Loaded 2021-22 gameweeks: 25447 rows
Loaded 2022-23 gameweeks: 26505 rows
Loaded 2023-24 gameweeks: 29725 rows
Loaded 2024-25 gameweeks: 27605 rows
Loaded 2025-26 gameweeks: 29757 rows
Loaded 2026-27 gameweeks: 29757 rows

Saved player_gw.csv: 193161 rows


In [3]:
hist_frames = []
for season in SEASONS:
    hist_path = RAW_DATA_DIR / season / "master_player_history.csv"
    if not hist_path.exists():
        continue

    df_h = pd.read_csv(hist_path, low_memory=False)
    df_h["source_season"] = season

    # Extract clean name from player_folder_name (e.g. "Mohamed_Salah_254" -> "mohamed salah")
    if "player_folder_name" in df_h.columns:
        raw_name = df_h["player_folder_name"].astype(str).str.rsplit("_", n=1).str[0].str.replace("_", " ")
        df_h["player_key"] = raw_name.apply(normalize_player_key)

    df_h["minutes"] = pd.to_numeric(df_h["minutes"], errors="coerce").fillna(0).clip(lower=0).astype(int)
    hist_frames.append(df_h)

player_history = pd.concat(hist_frames, ignore_index=True)
# Deduplicate by (element_code, season_name) keeping the latest record
if {"element_code", "season_name"}.issubset(player_history.columns):
    player_history = player_history.drop_duplicates(subset=["element_code", "season_name"], keep="last")

player_history.to_csv(PROCESSED_DIR / "player_history.csv", index=False)
print(f"Saved player_history.csv: {len(player_history)} rows")

Saved player_history.csv: 5705 rows


In [4]:
# 1. teams.csv (all canonical teams from clean_matches.csv)
clean_matches = pd.read_csv(PROCESSED_DIR / "clean_matches.csv")
unique_teams = sorted(set(clean_matches["Home_Team"]) | set(clean_matches["Away_Team"]))
teams_df = pd.DataFrame({
    "team_id": range(1, len(unique_teams) + 1),
    "team_name": unique_teams,
})
teams_df.to_csv(PROCESSED_DIR / "teams.csv", index=False)
print(f"Saved teams.csv: {len(teams_df)} teams")

# 2. players.csv (latest club, position, and value per unique player_key)
players_df = (
    player_gw.sort_values(["season", "round"])
    .groupby("player_key", as_index=False)
    .last()[["player_key", "name", "position", "team", "value", "season"]]
    .rename(columns={"season": "last_season"})
)
players_df.to_csv(PROCESSED_DIR / "players.csv", index=False)
print(f"Saved players.csv: {len(players_df)} unique players")

Saved teams.csv: 30 teams
Saved players.csv: 2124 unique players
